# EDA — RedRob AI Recruiter Challenge

In [ ]:
import json
import pandas as pd

DATA_PATH = '../[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge/sample_candidates.json'

with open(DATA_PATH) as f:
    candidates = json.load(f)

print(f'Candidate count: {len(candidates)}')

## Schema — top-level keys

In [ ]:
print('Top-level keys:', list(candidates[0].keys()))
print('profile keys:  ', list(candidates[0]['profile'].keys()))
print('signals keys:  ', list(candidates[0]['redrob_signals'].keys()))

## Skills structure

In [ ]:
# Skills are a list of objects — NOT ["Python", "SQL"] or {"Python": 5}
print('Skills type :', type(candidates[0]['skills']))
print('First skill :', candidates[0]['skills'][0])
print('Skill keys  :', list(candidates[0]['skills'][0].keys()))
# => name, proficiency (beginner/intermediate/advanced/expert), endorsements, duration_months
# Strategy: each skill has rich metadata -> use all 4 fields for feature engineering

## Flatten to DataFrame & missing values

In [ ]:
df = pd.json_normalize(candidates)
print('Shape:', df.shape)
df.isnull().sum()[df.isnull().sum() > 0]

## Behavioral signals — distributions

In [ ]:
signals = pd.json_normalize([c['redrob_signals'] for c in candidates])

key_signals = [
    'github_activity_score',       # -1 = no GitHub linked
    'recruiter_response_rate',
    'interview_completion_rate',
    'offer_acceptance_rate',       # -1 = no offer history
    'open_to_work_flag',
    'profile_completeness_score',
    'notice_period_days',
]

signals[key_signals].describe()

In [ ]:
# Sentinel values: -1 means 'not available', not a real score
print('github_activity_score == -1 :',  (signals['github_activity_score'] == -1).sum())
print('offer_acceptance_rate == -1 :', (signals['offer_acceptance_rate'] == -1).sum())
print('open_to_work_flag = True    :', signals['open_to_work_flag'].sum())

In [ ]:
# Salary range anomaly: some candidates have min > max
salary = pd.json_normalize([c['redrob_signals']['expected_salary_range_inr_lpa'] for c in candidates])
salary.columns = ['sal_min', 'sal_max']
inverted = salary[salary['sal_min'] > salary['sal_max']]
print(f'Candidates with sal_min > sal_max: {len(inverted)} / {len(salary)}')
print(inverted)

## Skill assessment scores — sparsity

In [ ]:
has_assessment = [bool(c['redrob_signals']['skill_assessment_scores']) for c in candidates]
print(f'Candidates with at least one assessment: {sum(has_assessment)} / {len(candidates)}')

# All assessed skills across the dataset
all_assessed = {}
for c in candidates:
    for skill, score in c['redrob_signals']['skill_assessment_scores'].items():
        all_assessed.setdefault(skill, []).append(score)

assessed_df = pd.DataFrame([
    {'skill': k, 'count': len(v), 'mean_score': round(sum(v)/len(v), 1)}
    for k, v in sorted(all_assessed.items(), key=lambda x: -len(x[1]))
])
print(assessed_df)

## Summary: feature engineering strategy

In [ ]:
print("""
KEY FINDINGS
============
1. Skills structure: list of objects with {name, proficiency, endorsements, duration_months}
   -> Feature options: skill match (name), weighted by proficiency level, endorsement count,
      and duration_months as a proxy for experience depth.

2. github_activity_score: -1 = no GitHub linked (treat as missing, not low score)
   offer_acceptance_rate: -1 = no offer history (impute or drop)

3. salary range: some candidates have min > max — needs cleaning before use.

4. skill_assessment_scores: sparse dict — only a minority of candidates have any.
   Use as a bonus signal, not a required feature.

5. open_to_work_flag: boolean — strong filter signal.
   recruiter_response_rate + interview_completion_rate: continuous engagement signals.
""")